# 04 - Construct core pre-treatment covariates

This notebook builds the first cell-level covariate table used by later estimation notebooks.

All covariates are aggregated to the same analysis grid used by Notebook 01 for deforestation. The intended production grid is `1km` by `1km`; if only legacy `0p080` files are present, the notebook can run in development mode but prints that clearly.

The central rule is that every covariate must be frozen before treatment exposure. For time-varying covariates, treated cells use only years strictly before their first treatment year. Never-treated cells use a common baseline window. Static geography covariates are time-invariant by construction.

Expected outputs:

- `data/intermediate/panel_core_covariates.parquet`
- `outputs/tables/04_covariate_balance_summary.csv`
- `outputs/tables/04_covariate_source_catalog.csv`


## Environment snapshot

In [1]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "project_config.json"
PROJECT_CONFIG = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}

RAW_DIR = Path(PROJECT_CONFIG.get("data_dirs", {}).get("raw", PROJECT_ROOT / "data" / "raw"))
INTERMEDIATE_DIR = Path(PROJECT_CONFIG.get("data_dirs", {}).get("intermediate", PROJECT_ROOT / "data" / "intermediate"))
PROCESSED_DIR = Path(PROJECT_CONFIG.get("data_dirs", {}).get("processed", PROJECT_ROOT / "data" / "processed"))
TABLE_DIR = Path(PROJECT_CONFIG.get("output_dirs", {}).get("tables", PROJECT_ROOT / "outputs" / "tables"))
FIGURE_DIR = Path(PROJECT_CONFIG.get("output_dirs", {}).get("figures", PROJECT_ROOT / "outputs" / "figures"))

for path in [RAW_DIR, INTERMEDIATE_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Intermediate dir:", INTERMEDIATE_DIR)
print("Tables dir:", TABLE_DIR)

Project root: c:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2
Intermediate dir: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\data\intermediate
Tables dir: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\outputs\tables


## User configuration

The `FREEZE_END_YEAR_NEVER_TREATED` setting is the common baseline cutoff for never-treated cells. Treated cells always use observations with `year < first_treat_year`. Static covariates ignore this year rule because their source is time-invariant or fixed at baseline.


In [ ]:
# --- Upstream inputs ---
# Production target: same 1km x 1km grid used for the deforestation panel in Notebook 01.
INTENDED_GRID_TAG = "1km"
ALLOW_LEGACY_GRID_FALLBACK = True
LEGACY_GRID_TAGS = ["0p080", ""]


def tagged_path(stem, suffix, tag):
    if tag:
        return INTERMEDIATE_DIR / f"{stem}_{tag}{suffix}"
    return INTERMEDIATE_DIR / f"{stem}{suffix}"


def resolve_grid_artifacts(target_tag=INTENDED_GRID_TAG):
    candidates = [target_tag]
    if ALLOW_LEGACY_GRID_FALLBACK:
        candidates += [tag for tag in LEGACY_GRID_TAGS if tag not in candidates]
    for tag in candidates:
        panel_path = tagged_path("panel_treatment", ".parquet", tag)
        grid_path = tagged_path("grid_geometry", ".geojson", tag)
        if panel_path.exists() and grid_path.exists():
            return tag, panel_path, grid_path
    tried = [(str(tagged_path("panel_treatment", ".parquet", tag)), str(tagged_path("grid_geometry", ".geojson", tag))) for tag in candidates]
    raise FileNotFoundError(f"No matching treatment/grid artifacts found. Tried: {tried}")

GRID_TAG, PANEL_TREATMENT_PATH, GRID_GEOMETRY_PATH = resolve_grid_artifacts()
USING_INTENDED_GRID = GRID_TAG == INTENDED_GRID_TAG

# --- Pre-treatment freezing rules ---
FREEZE_END_YEAR_NEVER_TREATED = 2013
MIN_PRE_YEARS_FOR_TRENDS = 3

# --- Earth Engine export controls ---
GEE_PROJECT = PROJECT_CONFIG.get("gee_project", "ee-cpedrazaj97")
RUN_GEE_EXPORTS = False
AUTO_START_EXPORT_TASKS = True
GEE_EXPORT_FOLDER = f"gee_covariates_core_{GRID_TAG}"
GEE_EXPORT_PREFIX = f"core_covariates_{GRID_TAG}"
GEE_SCALE_NTL = 1000
GEE_SCALE_TERRAIN = 90
GEE_SCALE_POPULATION = 100
GEE_TILE_SCALE = 16
N_GRID_CHUNKS = 64 if GRID_TAG == "1km" else 8
GRID_CHUNK_SEED = 42

# After Drive download, put the CSVs in this local directory.
LOCAL_GEE_COVARIATE_DIR = RAW_DIR / GEE_EXPORT_FOLDER
GEE_CONTINUOUS_GLOB = f"{GEE_EXPORT_PREFIX}_continuous_chunk*.csv"
GEE_POPULATION_GLOB = f"{GEE_EXPORT_PREFIX}_population_chunk*.csv"
# Backward-compatible single-file names, useful for small dev grids.
GEE_CONTINUOUS_CSV = LOCAL_GEE_COVARIATE_DIR / f"{GEE_EXPORT_PREFIX}_continuous.csv"
GEE_POPULATION_CSV = LOCAL_GEE_COVARIATE_DIR / f"{GEE_EXPORT_PREFIX}_population.csv"

# --- Outputs ---
CORE_COVARIATE_PATH = INTERMEDIATE_DIR / "panel_core_covariates.parquet"
BALANCE_SUMMARY_PATH = TABLE_DIR / "04_covariate_balance_summary.csv"
SOURCE_CATALOG_PATH = TABLE_DIR / "04_covariate_source_catalog.csv"

print("Intended grid tag:", INTENDED_GRID_TAG)
print("Active grid tag:", GRID_TAG)
print("Using intended production grid:", USING_INTENDED_GRID)
if not USING_INTENDED_GRID:
    print("WARNING: running with legacy/development grid artifacts. Re-run after Notebook 01/02 produce 1km artifacts.")
print("Treatment panel:", PANEL_TREATMENT_PATH)
print("Grid geometry:", GRID_GEOMETRY_PATH)
print("Common never-treated baseline ends in:", FREEZE_END_YEAR_NEVER_TREATED)
print("Run GEE exports:", RUN_GEE_EXPORTS)
print("GEE export chunks:", N_GRID_CHUNKS)

## Verify active grid support

The row unit must be the Notebook 01 grid cell. Earth Engine source pixels are reduced inside these grid polygons; they do not define the panel unit.


In [ ]:
grid_check_gdf = gpd.read_file(GRID_GEOMETRY_PATH)
if "cell_id" not in grid_check_gdf.columns:
    raise ValueError("Grid geometry is missing cell_id.")

area_km2 = grid_check_gdf.to_crs("EPSG:3857").geometry.area / 1e6
print("Grid cells:", f"{len(grid_check_gdf):,}")
print("Median cell area km2:", round(float(area_km2.median()), 3))
print("Cell area p05/p95 km2:", round(float(area_km2.quantile(0.05)), 3), round(float(area_km2.quantile(0.95)), 3))

if USING_INTENDED_GRID and not area_km2.median().between(0.5, 1.5):
    raise ValueError("Active grid tag is 1km, but median cell area is not close to 1 km2.")
if not USING_INTENDED_GRID:
    print("Development note: this is not the intended 1km production grid.")

## Covariate source catalog

This table is deliberately explicit. It records the source, unit, timing rule, and frozen output name before any data are merged.


In [ ]:
covariate_catalog = pd.DataFrame([
    {
        "covariate": "forest_loss_pre_mean_m2",
        "source": "Notebook 01 Hansen Global Forest Change panel, UMD/hansen/global_forest_change_2025_v1_13",
        "unit": "m2 forest loss per cell-year",
        "timing_rule": "Mean over years < first_treat_year for treated cells; mean over years <= FREEZE_END_YEAR_NEVER_TREATED for never-treated cells.",
        "status": "implemented_from_upstream_panel",
    },
    {
        "covariate": "forest_loss_pre_trend_m2_per_year",
        "source": "Notebook 01 Hansen Global Forest Change panel, UMD/hansen/global_forest_change_2025_v1_13",
        "unit": "m2 forest loss per year",
        "timing_rule": "OLS slope from pre-treatment forest-loss series; requires MIN_PRE_YEARS_FOR_TRENDS valid pre-treatment years.",
        "status": "implemented_from_upstream_panel",
    },
    {
        "covariate": "forest_loss_pre_sd_m2",
        "source": "Notebook 01 Hansen Global Forest Change panel, UMD/hansen/global_forest_change_2025_v1_13",
        "unit": "standard deviation of m2 forest loss per cell-year",
        "timing_rule": "Standard deviation over the same pre-treatment forest-loss window.",
        "status": "implemented_from_upstream_panel",
    },
    {
        "covariate": "dmsp_stable_lights_pre_mean",
        "source": "NOAA/DMSP-OLS/NIGHTTIME_LIGHTS, stable_lights band",
        "unit": "digital number, 0-63, cell mean",
        "timing_rule": "Mean over available annual DMSP years before treatment; never-treated cells use years <= FREEZE_END_YEAR_NEVER_TREATED.",
        "status": "gee_export_then_freeze",
    },
    {
        "covariate": "viirs_avg_rad_pre_mean",
        "source": "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG, avg_rad band",
        "unit": "nanoWatts/cm2/sr, annualized cell mean",
        "timing_rule": "Mean over available annual VIIRS years before treatment; cells treated before VIIRS begins remain missing.",
        "status": "gee_export_then_freeze",
    },
    {
        "covariate": "elevation_m_mean",
        "source": "USGS/SRTMGL1_003",
        "unit": "meters above sea level, cell mean",
        "timing_rule": "Static topography from SRTM; treated as pre-treatment geography.",
        "status": "gee_export_static",
    },
    {
        "covariate": "slope_deg_mean",
        "source": "USGS/SRTMGL1_003 transformed with ee.Terrain.slope",
        "unit": "degrees, cell mean",
        "timing_rule": "Static topography from SRTM; treated as pre-treatment geography.",
        "status": "gee_export_static",
    },
    {
        "covariate": "ruggedness_m_mean",
        "source": "USGS/SRTMGL1_003 local elevation standard deviation",
        "unit": "meters, cell mean",
        "timing_rule": "Static topography from SRTM; treated as pre-treatment geography.",
        "status": "gee_export_static",
    },
    {
        "covariate": "population_2000_sum",
        "source": "WorldPop/GP/100m/pop, Colombia image for year 2000",
        "unit": "estimated residents per grid cell",
        "timing_rule": "Baseline year 2000, before the forest-loss panel starts.",
        "status": "gee_export_baseline",
    },
    {
        "covariate": "friction_2019_mean",
        "source": "Oxford/MAP/friction_surface_2019",
        "unit": "travel impedance; higher values imply slower travel",
        "timing_rule": "Not pre-treatment for all designs. Use only as a geographic accessibility proxy or sensitivity check.",
        "status": "candidate_not_exported_by_default",
    },
    {
        "covariate": "distance_to_roads_km",
        "source": "External road vector source, preferably OpenStreetMap/official national road network, uploaded or processed outside GEE",
        "unit": "km from grid-cell centroid or cell polygon to nearest road",
        "timing_rule": "Use a baseline road vintage if available; otherwise label as time-invariant infrastructure proxy and test sensitivity.",
        "status": "external_needed",
    },
    {
        "covariate": "baseline_coca_presence_or_intensity",
        "source": "UNODC/SIMCI or Colombian coca-monitoring data if licensed/access is obtained",
        "unit": "indicator or hectares/intensity per cell",
        "timing_rule": "Use earliest pre-treatment year or pre-treatment mean; do not impute from post-treatment coca data.",
        "status": "external_restricted_not_currently_public",
    },
])

covariate_catalog.to_csv(SOURCE_CATALOG_PATH, index=False)
display(covariate_catalog)
print("Saved catalog:", SOURCE_CATALOG_PATH)

## Load treatment panel and freeze forest-loss history

Forest-loss covariates come from Notebook 01 / Notebook 02 outputs. They are cell-level summaries and are not allowed to use treated or post-treatment years.


In [ ]:
if not PANEL_TREATMENT_PATH.exists():
    raise FileNotFoundError(PANEL_TREATMENT_PATH)

panel = pd.read_parquet(PANEL_TREATMENT_PATH)
required_cols = {"cell_id", "year", "loss_m2", "first_treat_year", "ever_treated", "never_treated"}
missing = required_cols.difference(panel.columns)
if missing:
    raise ValueError(f"Treatment panel is missing required columns: {sorted(missing)}")

cell_timing = (
    panel[["cell_id", "first_treat_year", "ever_treated", "never_treated", "cell_lon", "cell_lat", "base_m2"]]
    .drop_duplicates("cell_id")
    .copy()
)

panel_for_freeze = panel[["cell_id", "year", "loss_m2", "first_treat_year", "ever_treated", "never_treated"]].copy()
panel_for_freeze["freeze_cutoff_year"] = np.where(
    panel_for_freeze["ever_treated"].eq(1),
    panel_for_freeze["first_treat_year"].astype("float") - 1,
    FREEZE_END_YEAR_NEVER_TREATED,
)
panel_pre = panel_for_freeze[panel_for_freeze["year"] <= panel_for_freeze["freeze_cutoff_year"]].copy()

print("Cells:", cell_timing["cell_id"].nunique())
print("Panel years:", int(panel["year"].min()), "to", int(panel["year"].max()))
print("Pre-treatment forest rows retained:", f"{len(panel_pre):,}")

In [ ]:
def slope_or_nan(years, values, min_obs=3):
    mask = np.isfinite(years) & np.isfinite(values)
    years = np.asarray(years)[mask]
    values = np.asarray(values)[mask]
    if len(years) < min_obs or np.nanstd(years) == 0:
        return np.nan
    x = years - years.mean()
    return float(np.sum(x * (values - values.mean())) / np.sum(x * x))

forest_covariates = (
    panel_pre.groupby("cell_id", as_index=False)
    .agg(
        forest_loss_pre_years=("year", "nunique"),
        forest_loss_pre_mean_m2=("loss_m2", "mean"),
        forest_loss_pre_sd_m2=("loss_m2", "std"),
        forest_loss_pre_total_m2=("loss_m2", "sum"),
    )
)

forest_trends = (
    panel_pre.groupby("cell_id")
    .apply(lambda x: slope_or_nan(x["year"].to_numpy(dtype=float), x["loss_m2"].to_numpy(dtype=float), MIN_PRE_YEARS_FOR_TRENDS), include_groups=False)
    .rename("forest_loss_pre_trend_m2_per_year")
    .reset_index()
)

forest_covariates = forest_covariates.merge(forest_trends, on="cell_id", how="left")
forest_covariates["forest_loss_pre_sd_m2"] = forest_covariates["forest_loss_pre_sd_m2"].fillna(0)

print(forest_covariates.shape)
display(forest_covariates.head())

## Optional Earth Engine export: night lights, terrain, and population

Run this section only when you want to create or refresh the GEE-derived covariate CSVs. It uses the exact grid geometry from Notebook 01, converts it to an Earth Engine feature collection, and exports one row per grid cell. For the 1km grid, exports are chunked so Earth Engine does not need to write the whole national table in one task.

Sources checked against the Earth Engine catalog:

- DMSP yearly nighttime lights: `NOAA/DMSP-OLS/NIGHTTIME_LIGHTS`
- VIIRS monthly nighttime lights: `NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG`
- SRTM elevation: `USGS/SRTMGL1_003`
- WorldPop population: `WorldPop/GP/100m/pop`


In [ ]:
if RUN_GEE_EXPORTS:
    import ee
    import geemap

    try:
        ee.Initialize(project=GEE_PROJECT)
        print("Earth Engine initialized successfully.")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
        print("Earth Engine initialized successfully after authentication.")

    grid_gdf = gpd.read_file(GRID_GEOMETRY_PATH)[["cell_id", "geometry"]].copy()
    grid_ee = geemap.gdf_to_ee(grid_gdf)

    def add_grid_chunk(f):
        chunk = ee.Number(f.get("grid_random")).multiply(N_GRID_CHUNKS).floor().int()
        return f.set("grid_chunk", chunk)

    grid_ee = grid_ee.randomColumn("grid_random", seed=GRID_CHUNK_SEED).map(add_grid_chunk)
    print("Grid cells sent to Earth Engine:", len(grid_gdf))
    print("Chunk count:", N_GRID_CHUNKS)

In [ ]:
if RUN_GEE_EXPORTS:
    def dmsp_annual_band(year):
        img = (
            ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS")
            .filter(ee.Filter.calendarRange(year, year, "year"))
            .select("stable_lights")
            .mean()
            .rename(f"dmsp_stable_lights_{year}")
        )
        return img

    def viirs_annual_band(year):
        img = (
            ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .select("avg_rad")
            .mean()
            .rename(f"viirs_avg_rad_{year}")
        )
        return img

    srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")
    elevation = srtm.rename("elevation_m")
    slope = ee.Terrain.slope(srtm).rename("slope_deg")
    ruggedness = srtm.reduceNeighborhood(
        reducer=ee.Reducer.stdDev(),
        kernel=ee.Kernel.square(radius=1, units="pixels"),
    ).rename("ruggedness_m")

    dmsp_bands = [dmsp_annual_band(y) for y in range(2001, 2014)]
    viirs_bands = [viirs_annual_band(y) for y in range(2012, 2026)]
    continuous_img = ee.Image.cat([elevation, slope, ruggedness] + dmsp_bands + viirs_bands)

    continuous_tasks = {}
    for chunk in range(N_GRID_CHUNKS):
        chunk_fc = grid_ee.filter(ee.Filter.eq("grid_chunk", chunk))
        reduced = continuous_img.reduceRegions(
            collection=chunk_fc,
            reducer=ee.Reducer.mean(),
            scale=GEE_SCALE_TERRAIN,
            tileScale=GEE_TILE_SCALE,
        )
        description = f"{GEE_EXPORT_PREFIX}_continuous_chunk{chunk:02d}"
        task = ee.batch.Export.table.toDrive(
            collection=reduced,
            description=description,
            folder=GEE_EXPORT_FOLDER,
            fileNamePrefix=description,
            fileFormat="CSV",
        )
        continuous_tasks[description] = task
        if AUTO_START_EXPORT_TASKS:
            task.start()
    print(("Started" if AUTO_START_EXPORT_TASKS else "Prepared"), len(continuous_tasks), "continuous covariate export task(s).")

In [ ]:
if RUN_GEE_EXPORTS:
    pop2000 = (
        ee.ImageCollection("WorldPop/GP/100m/pop")
        .filter(ee.Filter.eq("country", "COL"))
        .filter(ee.Filter.eq("year", 2000))
        .first()
        .select("population")
        .rename("population_2000")
    )

    population_tasks = {}
    for chunk in range(N_GRID_CHUNKS):
        chunk_fc = grid_ee.filter(ee.Filter.eq("grid_chunk", chunk))
        reduced = pop2000.reduceRegions(
            collection=chunk_fc,
            reducer=ee.Reducer.sum(),
            scale=GEE_SCALE_POPULATION,
            tileScale=GEE_TILE_SCALE,
        )
        description = f"{GEE_EXPORT_PREFIX}_population_chunk{chunk:02d}"
        task = ee.batch.Export.table.toDrive(
            collection=reduced,
            description=description,
            folder=GEE_EXPORT_FOLDER,
            fileNamePrefix=description,
            fileFormat="CSV",
        )
        population_tasks[description] = task
        if AUTO_START_EXPORT_TASKS:
            task.start()
    print(("Started" if AUTO_START_EXPORT_TASKS else "Prepared"), len(population_tasks), "population covariate export task(s).")

## Load GEE covariate exports if available

This block is intentionally tolerant: notebook 04 can already build the forest-history covariates before the GEE CSVs are downloaded. Once the CSVs exist locally, rerun from here to merge and freeze them.


In [ ]:
def read_gee_csvs_if_available(directory, glob_pattern, fallback_path=None):
    directory = Path(directory)
    files = sorted(directory.glob(glob_pattern)) if directory.exists() else []
    if not files and fallback_path is not None and Path(fallback_path).exists():
        files = [Path(fallback_path)]
    if not files:
        print("Missing optional GEE export(s):", directory / glob_pattern)
        return None

    frames = []
    for path in files:
        df = pd.read_csv(path)
        if "cell_id" not in df.columns:
            raise ValueError(f"{path} does not contain cell_id")
        drop_cols = [c for c in ["system:index", ".geo", "id", "grid_random", "grid_chunk"] if c in df.columns]
        if drop_cols:
            df = df.drop(columns=drop_cols)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates("cell_id", keep="last")
    print("Loaded", len(files), "file(s) with", f"{len(out):,}", "unique cells from", glob_pattern)
    return out

continuous_gee = read_gee_csvs_if_available(LOCAL_GEE_COVARIATE_DIR, GEE_CONTINUOUS_GLOB, GEE_CONTINUOUS_CSV)
population_gee = read_gee_csvs_if_available(LOCAL_GEE_COVARIATE_DIR, GEE_POPULATION_GLOB, GEE_POPULATION_CSV)

gee_covariates = cell_timing[["cell_id"]].copy()
if continuous_gee is not None:
    gee_covariates = gee_covariates.merge(continuous_gee, on="cell_id", how="left")
if population_gee is not None:
    pop_cols = ["cell_id"] + [c for c in population_gee.columns if c != "cell_id"]
    population_gee = population_gee[pop_cols].rename(columns={"population_2000": "population_2000_sum"})
    gee_covariates = gee_covariates.merge(population_gee, on="cell_id", how="left")

print(gee_covariates.shape)
display(gee_covariates.head())

## Freeze time-varying GEE covariates

DMSP and VIIRS are annual series. This block converts annual columns into frozen pre-treatment summaries. Static terrain and baseline population are kept as cell-level values.


In [ ]:
def find_year_columns(df, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d{{4}})$")
    pairs = []
    for col in df.columns:
        match = pattern.match(col)
        if match:
            pairs.append((int(match.group(1)), col))
    return sorted(pairs)

def frozen_row_mean(row, year_cols, first_treat_year, ever_treated, never_cutoff):
    values = []
    for year, col in year_cols:
        if ever_treated == 1:
            keep = pd.notna(first_treat_year) and year < int(first_treat_year)
        else:
            keep = year <= never_cutoff
        if keep:
            values.append(row.get(col, np.nan))
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    return float(values.mean()) if values.notna().any() else np.nan

gee_frozen = gee_covariates.merge(
    cell_timing[["cell_id", "first_treat_year", "ever_treated"]],
    on="cell_id",
    how="left",
)

dmsp_cols = find_year_columns(gee_frozen, "dmsp_stable_lights")
viirs_cols = find_year_columns(gee_frozen, "viirs_avg_rad")

if dmsp_cols:
    gee_frozen["dmsp_stable_lights_pre_mean"] = gee_frozen.apply(
        lambda r: frozen_row_mean(r, dmsp_cols, r["first_treat_year"], r["ever_treated"], FREEZE_END_YEAR_NEVER_TREATED),
        axis=1,
    )
    gee_frozen["dmsp_stable_lights_pre_years"] = gee_frozen.apply(
        lambda r: sum((y < int(r["first_treat_year"]) if r["ever_treated"] == 1 and pd.notna(r["first_treat_year"]) else y <= FREEZE_END_YEAR_NEVER_TREATED) and pd.notna(r.get(c)) for y, c in dmsp_cols),
        axis=1,
    )

if viirs_cols:
    gee_frozen["viirs_avg_rad_pre_mean"] = gee_frozen.apply(
        lambda r: frozen_row_mean(r, viirs_cols, r["first_treat_year"], r["ever_treated"], FREEZE_END_YEAR_NEVER_TREATED),
        axis=1,
    )
    gee_frozen["viirs_avg_rad_pre_years"] = gee_frozen.apply(
        lambda r: sum((y < int(r["first_treat_year"]) if r["ever_treated"] == 1 and pd.notna(r["first_treat_year"]) else y <= FREEZE_END_YEAR_NEVER_TREATED) and pd.notna(r.get(c)) for y, c in viirs_cols),
        axis=1,
    )

static_cols = [c for c in ["elevation_m", "slope_deg", "ruggedness_m", "population_2000_sum"] if c in gee_frozen.columns]
rename_static = {
    "elevation_m": "elevation_m_mean",
    "slope_deg": "slope_deg_mean",
    "ruggedness_m": "ruggedness_m_mean",
}

keep_gee_cols = ["cell_id"] + static_cols + [
    c for c in [
        "dmsp_stable_lights_pre_mean",
        "dmsp_stable_lights_pre_years",
        "viirs_avg_rad_pre_mean",
        "viirs_avg_rad_pre_years",
    ] if c in gee_frozen.columns
]
gee_frozen = gee_frozen[keep_gee_cols].rename(columns=rename_static)

print(gee_frozen.shape)
display(gee_frozen.head())

## Assemble core covariate table

The table is one row per grid cell. Estimation notebooks can merge it onto the panel by `cell_id`.


In [ ]:
core_covariates = (
    cell_timing
    .merge(forest_covariates, on="cell_id", how="left")
    .merge(gee_frozen, on="cell_id", how="left")
)

# Drop duplicate columns if an optional export was not loaded cleanly.
core_covariates = core_covariates.loc[:, ~core_covariates.columns.duplicated()].copy()

ordered_front = [
    "cell_id", "cell_lon", "cell_lat", "base_m2",
    "first_treat_year", "ever_treated", "never_treated",
]
other_cols = [c for c in core_covariates.columns if c not in ordered_front]
core_covariates = core_covariates[ordered_front + other_cols]

core_covariates.to_parquet(CORE_COVARIATE_PATH, index=False)

print("Saved:", CORE_COVARIATE_PATH)
print(core_covariates.shape)
display(core_covariates.head())

## Balance summary by ever-treated status

This is a descriptive check, not a matching or weighting procedure. The standardized difference is computed as the treated-minus-never difference divided by the pooled standard deviation.


In [ ]:
def balance_summary(df, group_col="ever_treated"):
    excluded = {"cell_id", "cell_lon", "cell_lat", "first_treat_year", "ever_treated", "never_treated"}
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in excluded]
    rows = []
    for col in numeric_cols:
        treated = pd.to_numeric(df.loc[df[group_col].eq(1), col], errors="coerce").dropna()
        control = pd.to_numeric(df.loc[df[group_col].eq(0), col], errors="coerce").dropna()
        t_mean = treated.mean() if len(treated) else np.nan
        c_mean = control.mean() if len(control) else np.nan
        t_sd = treated.std(ddof=1) if len(treated) > 1 else np.nan
        c_sd = control.std(ddof=1) if len(control) > 1 else np.nan
        pooled = np.sqrt((t_sd**2 + c_sd**2) / 2) if pd.notna(t_sd) and pd.notna(c_sd) else np.nan
        std_diff = (t_mean - c_mean) / pooled if pooled and pooled > 0 else np.nan
        rows.append({
            "covariate": col,
            "n_ever_treated": len(treated),
            "mean_ever_treated": t_mean,
            "sd_ever_treated": t_sd,
            "n_never_treated": len(control),
            "mean_never_treated": c_mean,
            "sd_never_treated": c_sd,
            "difference": t_mean - c_mean if pd.notna(t_mean) and pd.notna(c_mean) else np.nan,
            "standardized_difference": std_diff,
            "missing_share": df[col].isna().mean(),
        })
    return pd.DataFrame(rows)

balance = balance_summary(core_covariates)
balance.to_csv(BALANCE_SUMMARY_PATH, index=False)

print("Saved:", BALANCE_SUMMARY_PATH)
display(balance.sort_values("standardized_difference", key=lambda s: s.abs(), ascending=False).head(30))

## Notes on additional covariates

Good candidates for this notebook, in roughly decreasing priority:

1. `distance_to_roads_km`: use OpenStreetMap or an official Colombian road network outside GEE, then compute nearest-distance locally in a projected CRS. This is more transparent than relying on a stale or partial GEE road layer.
2. `distance_to_settlements_km` or `baseline_population_2000_sum`: population is already available through WorldPop; settlement distance can be added from GHSL built-up polygons or official settlement points.
3. `baseline_coca_presence_or_intensity`: likely needs licensed UNODC/SIMCI data or a formal request. If only department/municipality aggregates are available, keep it at that geography and label the unit clearly.
4. `pre_treatment_precipitation_mean` and `pre_treatment_temperature_mean`: useful if ecological productivity or accessibility varies strongly across cells; CHIRPS precipitation is straightforward in GEE.
5. `baseline_land_cover_shares`: MODIS MCD12Q1 classes can capture cropland, savanna, urban, and wetland shares around 2001.

Do not mix pre- and post-treatment vintages silently. If a variable is measured after some cells are treated, either exclude it from the main covariate set or label it as a sensitivity-only proxy.
